In [1]:
# preprocess import numpy as np
from scipy.interpolate import griddata
from datetime import timedelta, datetime, date
import dateutil.parser
import pandas as pd
from scipy.spatial import distance_matrix
import xarray as xr
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
def creating_geotiff(file_netcdf,country,n_month):
    bT = file_netcdf['ssrd_integrated']
    bT = bT.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
    bT.rio.crs
    # Define the CRS projection
    bT = bT.rio.write_crs("epsg:4326", inplace=True)
    bT.rio.to_raster(output_path+f'/ssrd_integrated_{country}_m{n_month}.tif')
    return bT

## Integration of the downscaled data with the stations data (Croatia use case)

In [3]:
## 
country = 'croatia' ## FILL WITH THE COUNTRY NAME e.g.croatia

In [4]:
month = 1

In [6]:
## 
data_path = ''## FILL WITH THE PATH of STATIONS data
stations_path = f'{data_path}allstat_solar_radiation.csv' 
downscaled_path = f'../data/downscaled_data/{country}/' ## FILL WITH THE PATH of ERA5LAND data
idw_path = f'../data/idw_data/{country}/' ## FILL WITH THE PATH of ERA5LAND data
output_path = f'../data/atlas_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [9]:
def opening_datasets(data_path, n_month, country):
    era5_ds = xr.open_dataset(downscaled_path+f'ssrd_downscaled_{country}_m{n_month}.nc').sortby('latitude')
    #Latitudes have to be in descending order
    era5_ds = era5_ds.reindex(latitude = era5_ds.latitude[::-1])

    idw_ds  = xr.open_dataset(idw_path+f'ssrd_idw_{country}_m{n_month}.nc')
    
    return era5_ds, idw_ds

In [10]:
era5_ds, idw_ds = opening_datasets(data_path,month,country)

$$
\Huge
ssrd_\mathrm{integrated} = \frac{\frac{\mathrm{ssrd}_{\mathrm{era5}}}{\sigma_{\mathrm{era5}}^2} + \frac{\mathrm{ssrd}_{\mathrm{idw}}}{\sigma_{\mathrm{idw}}^2}}{\frac{1}{\sigma_{\mathrm{era5}}^2} + \frac{1}{\sigma_{\mathrm{idw}}^2}}
$$

In [11]:
def apply_dist(pdf, lat_new, lon_new):

    lat = pdf['latitude'].drop_duplicates().values
    lon = pdf['longitude'].drop_duplicates().values

    # Locations of the IoT observations
    iot_grid = np.array(list(zip(lat, lon)))

    # Grid for IoT observation projection
    grid_new = np.array([[x, y] for x in lat_new for y in lon_new])

    # Calculating IoT map
    iot_new = distances(iot_grid, grid_new)

    # Reshaping results from array-like to matrix-like
    iot_reshape = []
    for i in np.arange(0,len(iot_new)):
        iot_reshape.append(iot_new[i].reshape(len(lat_new), len(lon_new)))

    return iot_reshape

In [12]:
def distances(pos, posi):

    # Distance matrix - this is part of one of the functions used in the IDW
    dist = distance_matrix(pos, posi)**3         

    return dist

In [13]:
def min_distances(idw_ds,data_path,country):
    lat_fin = idw_ds.latitude.to_numpy()
    lon_fin = idw_ds.longitude.to_numpy()
    stats_data = pd.read_csv(stations_path)
    dist_matrices = apply_dist(stats_data, lat_fin, lon_fin)
    
    for i in np.arange(1,len(dist_matrices)):
        if i == 1:
            min_init = np.minimum(dist_matrices[i],dist_matrices[i-1])
        else:
            min_dist = np.minimum(min_init,dist_matrices[i])
        
    min_dist_xr  = xr.DataArray(min_dist, coords=[("latitude", lat_fin), ("longitude", lon_fin)])

    # NORMALIZATION 
    min_dist_xr_norm = (min_dist_xr - min_dist_xr.min()) / (min_dist_xr.max() - min_dist_xr.min())
    return min_dist_xr_norm

In [14]:
min_dist_xr_norm = min_distances(idw_ds,data_path,country)

In [15]:
def uncertainties(min_dist_xr_norm):
    #uncertainties of the era5 dataset
    sigma_2_era5_norm= (0.2)**2
    
    #uncertainties of the idw dataset
    sigma_2_idw_norm = (0.4 + min_dist_xr_norm)**2
    
    return sigma_2_era5_norm, sigma_2_idw_norm

In [16]:
sigma_2_era5_norm, sigma_2_idw_norm = uncertainties(min_dist_xr_norm)

In [17]:
def apply_scaling(era5_ds, sigma_2_era5_norm, idw_ds, sigma_2_idw_norm,n_month,country):
    var_scaled = ((era5_ds.ssrd_downscaled / sigma_2_era5_norm) +
                  (idw_ds.ssrd_reshaped / sigma_2_idw_norm)) / ((1/sigma_2_era5_norm) + (1/sigma_2_idw_norm))
    
    var_scaled_xr = var_scaled.to_dataset(dim=None, name='ssrd_integrated')
    var_scaled_xr.to_netcdf(output_path+f'ssrd_integrated_{country}_m{n_month}.nc')
    
    geotiff_file = creating_geotiff(var_scaled_xr,country,n_month)
    return var_scaled_xr

In [18]:
ssrd_integrated_xr = apply_scaling(era5_ds, sigma_2_era5_norm, idw_ds, sigma_2_idw_norm,month,country)

## Integration of the downscaled data with the satellite data validated on stations data (Cubause case)

In [3]:
## 
country = 'cuba' ## FILL WITH THE COUNTRY NAME e.g. cuba

In [4]:
## 
downscaled_path = f'../data/downscaled_data/{country}/' ## FILL WITH THE PATH of ERA5LAND data
data_path = ''## FILL WITH THE PATH of SATELLITE data
sat_path = f'{data_path}heliosat_stations_validated.nc' 
output_path = f'../data/atlas_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [5]:
month = 1

In [6]:
sigma_2_era5_norm= (0.2)**2
sigma_2_idw_norm = (0.4)**2

In [10]:
def opening_datasets(n_month, country):
    era5_ds = xr.open_dataset(downscaled_path+f'ssrd_downscaled_{country}_m{n_month}.nc').sortby('latitude')
    months_dic = {'1':'january','2':'february','3':'march','4':'april','5':'may','6':'june','7':'july',
                  '8':'august','9':'september','10':'october','11':'november','12':'december'}
    sat_ds  = xr.open_mfdataset(sat_path)[months_dic[f'{n_month}']].load().to_dataset(name='ssrd_reshaped')
    return era5_ds, sat_ds

$$
\Huge
ssrd_\mathrm{integrated} = \frac{\frac{\mathrm{ssrd}_{\mathrm{era5}}}{\sigma_{\mathrm{era5}}^2} + \frac{\mathrm{ssrd}_{\mathrm{idw}}}{\sigma_{\mathrm{idw}}^2}}{\frac{1}{\sigma_{\mathrm{era5}}^2} + \frac{1}{\sigma_{\mathrm{idw}}^2}}
$$

In [8]:
def apply_scaling(era5_ds, sigma_2_era5_norm, idw_ds, sigma_2_idw_norm, n_month,country):
    var_scaled = ((era5_ds.ssrd_downscaled / sigma_2_era5_norm) +
                  (idw_ds.ssrd_reshaped / sigma_2_idw_norm)) / ((1/sigma_2_era5_norm) + (1/sigma_2_idw_norm))
    
    var_scaled_xr = var_scaled.to_dataset(dim=None, name='ssrd_integrated')
    var_scaled_xr.to_netcdf(output_path+f'ssrd_integrated_{country}_m{n_month}.nc')
    
    geotiff_file = creating_geotiff(var_scaled_xr,country,n_month)
    
    return var_scaled_xr

In [11]:
era5_ds, idw_ds = opening_datasets(month,country)
var_integrated_xr = apply_scaling(era5_ds, sigma_2_era5_norm, idw_ds, sigma_2_idw_norm, month,country)